In [2]:
import time
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [3]:
random_forest_features = pd.read_csv("../results/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/mrmr_10_features.csv")["feature"].tolist()

features_map = {}

features_map["Random Forest"] = random_forest_features
features_map["Correlation"] = correlation_features[:10]  # Limiting to top 10 features
features_map["Gevrey Method (10 features)"] = gevrey_method_features[:10]  # Limiting to top 10 features
features_map["Gevrey Method (14 features)"] = gevrey_method_features[:14]  # Limiting to top 14 features
features_map["mRMR (10 features)"] = mrmr_10_features

In [4]:
class DatasetScalerService:
    MAX_LIMIT = 100_000
    OFFSET    = 1_000

    def __init__(self, features: list[str]):
        self.__scaler_X = StandardScaler()
        self.__X_original = pd.read_csv("../dataset/j_kampe.csv")
        self.__y_original = pd.read_csv("../dataset/distances.csv")["distance"]
        self.__features = features

    def get_scaled_data(self, limit: int = 11_000):
        if limit + self.OFFSET > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.__X_original[self.__features].values
        y = self.__y_original.values.ravel()

        X = X[self.OFFSET:limit + self.OFFSET]
        y = y[self.OFFSET:limit + self.OFFSET]

        X_train = X[:int(0.8 * X.shape[0])]
        X_test  = X[int(0.8 * X.shape[0]):]
        y_train = y[:int(0.8 * y.shape[0])]
        y_test  = y[int(0.8 * y.shape[0]):]

        X_train_scaled = self.__scaler_X.fit_transform(X_train)
        X_test_scaled  = self.__scaler_X.transform(X_test)
        return X_train_scaled, X_test_scaled, y_train, y_test

In [5]:
def create_quantum_kernel(n_qubits, reps=2, entanglement="linear"):
    feature_map = pauli_feature_map(feature_dimension=n_qubits, reps=reps, entanglement=entanglement)
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    return quantum_kernel

In [6]:
results = []
C = 10
epsilon = 0.001

for feature_name, features in features_map.items():
    n_qubits = len(features)

    print(f"Running QSVR experiment with feature set: {feature_name} ({n_qubits} features)")

    dataset_service = DatasetScalerService(features)
    X_train, X_test, y_train, y_test = dataset_service.get_scaled_data(limit=1_000)

    quantum_kernel = create_quantum_kernel(n_qubits=n_qubits)

    svr = SVR(kernel=quantum_kernel.evaluate, C=C, epsilon=epsilon)

    start = time.time()
    svr.fit(X_train, y_train)
    y_pred = svr.predict(X_test)
    end = time.time()
    elapsed_time = end - start

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_name": feature_name,
        "rmse": rmse,
        "r2": r2,
        "n_qubits": n_qubits,
        "elapsed_time": elapsed_time
    })

results_df = pd.DataFrame(results)
results_df.to_csv("../results/qsvr_v2_experiment_8_paulifeaturemap_results.csv", index=False)
print(results_df)

Running QSVR experiment with feature set: Random Forest (5 features)
Running QSVR experiment with feature set: Correlation (10 features)
Running QSVR experiment with feature set: Gevrey Method (10 features) (10 features)
Running QSVR experiment with feature set: Gevrey Method (14 features) (14 features)
Running QSVR experiment with feature set: mRMR (10 features) (10 features)
                  feature_name      rmse         r2  n_qubits  elapsed_time
0                Random Forest  1.719437 -32.344969         5   3297.697420
1                  Correlation  0.302314  -0.030802        10   6976.476178
2  Gevrey Method (10 features)  0.300584  -0.019034        10   6965.121763
3  Gevrey Method (14 features)  0.298944  -0.007944        14  39371.784992
4           mRMR (10 features)  0.296790   0.006528        10   6839.397648
